# Structured streaming Demo

In [7]:
# Imports cell

from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    abs as abs_col,
    avg,
    col,
    count,
    from_json,
    from_unixtime,
    max as max_col,
    sum as sum_col,
    to_timestamp,
    window,
)
from pyspark.sql.types import (
    BooleanType,
    IntegerType,
    LongType,
    StringType,
    StructField,
    StructType,
)



In [8]:
# initialize the spark session
spark = SparkSession.builder \
    .appName("KafkaStreamingPrototype") \
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.13:4.1.0") \
    .getOrCreate()


In [9]:
# Define the initial raw streaming dataframe
raw_df = spark \
    .readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "kafka:19092") \
    .option("subscribe", "wikistreams") \
    .option("startingOffsets", "earliest") \
    .load()


In [10]:
# Define a simplified schema - just the key fields for analysis
simple_wikimedia_schema = StructType([
    StructField("title", StringType(), True),
    StructField("user", StringType(), True),
    StructField("timestamp", LongType(), True),
    StructField("type", StringType(), True),
    StructField("wiki", StringType(), True),
    StructField("bot", BooleanType(), True),
    StructField("minor", BooleanType(), True),
    StructField("comment", StringType(), True),
    StructField("length", StructType([
        StructField("old", IntegerType(), True),
        StructField("new", IntegerType(), True)
    ]), True)
])

# Apply the schema to organize the raw input
parsed_df = raw_df.select(
    from_json(col("value").cast("string"), simple_wikimedia_schema).alias("data")
).select("data.*")

In [11]:
# Adjust these values to change how frequently the aggregates update.
COUNT_WINDOW_DURATION = "5 minutes"
COUNT_WINDOW_SLIDE = "1 minute"
SIZE_WINDOW_DURATION = "10 minutes"

# Convert the event timestamp to Spark timestamp and measure the page-size delta.
# Positive changes mean the page grew; negative changes mean it shrank.
edits_df = parsed_df \
    .withColumn("event_time", to_timestamp(from_unixtime(col("timestamp")))) \
    .withColumn(
        "signed_size_change",
        col("length.new") - col("length.old")
    ) \
    .withColumn("edit_size_bytes", abs_col(col("signed_size_change")))

# OVERLAPPING (sliding) windows: each edit can appear in five one-minute steps
# because the five-minute window advances every minute.
windowed_counts_df = edits_df \
    .groupBy(
        window(col("event_time"), COUNT_WINDOW_DURATION, COUNT_WINDOW_SLIDE),
        col("bot")
    ) \
    .agg(count("*").alias("edit_count")) \
    .select(
        col("window.start").alias("window_start"),
        col("window.end").alias("window_end"),
        col("bot"),
        col("edit_count")
    )

# TUMBLING windows: each edit belongs to one non-overlapping ten-minute window.
# Omitting the slide duration makes it equal to the window duration.
# Size fields can be missing, so count them separately from all edits.
windowed_edit_sizes_df = edits_df \
    .groupBy(
        window(col("event_time"), SIZE_WINDOW_DURATION),
        col("wiki"),
        col("bot")
    ) \
    .agg(
        count("*").alias("edit_count"),
        count("edit_size_bytes").alias("edits_with_size_data"),
        avg("edit_size_bytes").alias("average_size_change_bytes"),
        max_col("edit_size_bytes").alias("largest_size_change_bytes"),
        sum_col("signed_size_change").alias("net_size_change_bytes")
    ) \
    .select(
        col("window.start").alias("window_start"),
        col("window.end").alias("window_end"),
        col("wiki"),
        col("bot"),
        col("edit_count"),
        col("edits_with_size_data"),
        col("average_size_change_bytes"),
        col("largest_size_change_bytes"),
        col("net_size_change_bytes")
    )

In [12]:

# Both console sinks write to Docker logs; use distinct query names to tell them apart.
edit_counts_query = windowed_counts_df \
    .writeStream \
    .format("console") \
    .outputMode("complete") \
    .option("truncate", False) \
    .queryName("wiki_edit_counts") \
    .start()

edit_sizes_query = windowed_edit_sizes_df \
    .writeStream \
    .format("console") \
    .outputMode("complete") \
    .option("truncate", False) \
    .queryName("wiki_edit_sizes") \
    .start()